# Streaming Timepix3 events to parquet, and into Tiled — demonstration

This notebook demonstrates the three criteria set for the parquet output of `splash_timepix`,
one section each, headed with the criterion verbatim:

1. **Data Streaming of timepix3 into parquet files**
2. **Tiled should be able to read the file**
3. **Taking metadata out of files, and having tiled put that metadata into its database**

It generates its own data with the repository's simulated detector (`tpx-sim`), runs the real
streaming server (`tpx-stream`, the process the UI launches) and a real Tiled server
(`tiled serve catalog`) on free ports under a temporary directory. No detector, no `live-cli`,
nothing beyond localhost; **Run all** takes about one minute. The process plumbing is in
[`demo_helpers.py`](demo_helpers.py) so the cells show only what is being checked.

The notebook is kept executed, so it reads as evidence without being run. Requirements:
`pip install -e ".[tiled-test]"` (Tiled server + client, dask, pandas). Manual, format and
server set-up: [`../README.md`](../README.md), [`../format.md`](../format.md),
[`../tiled.md`](../tiled.md).

In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from tiled.queries import Key

from demo_helpers import Demo, tree

pd.set_option("display.width", 140)
demo = Demo()
print("working directory:", demo.workdir)
print("Tiled will listen on:", demo.tiled_uri)

working directory: /tmp/tpx3_demo_1pmjo2i3
Tiled will listen on: http://127.0.0.1:59697


## 1. "Data Streaming of timepix3 into parquet files"

`tpx-stream --parquet-dir <output dir>` writes every pixel hit, TDC edge and control packet of
the scan to `<output dir>/<scan_uuid>/{events,tdc,control}/part-NNNNNN.parquet` **while the
acquisition runs**, next to the ZMQ live view it has always published. The operator fields
(sample, ESAF, notes) travel through the same metadata file the UI writes.

Here the simulated detector streams at 15 k hits/s for 20 s. The roll size is set to 0.25 MB
(default 512 MB) so that files appear every few seconds at this modest rate.

In [2]:
demo.start_stream(roll_mb=0.25, sample="Cu foil (demo)", esaf="demo-0001")
demo.start_simulator(cps=15_000, duration=20)
scan_dir = demo.wait_scan_dir()
print("scan directory created by tpx-stream:", scan_dir)

2026-10-21 13:05:50,890 - INFO - Server ready (state: ready)


scan directory created by tpx-stream: /tmp/tpx3_demo_1pmjo2i3/data/a8214d88-e269-482e-bbef-cd2f311f4ff8


**Files appear while data is still flowing.** Two listings of the scan directory, eight seconds apart:

In [3]:
time.sleep(6)
print(time.strftime("%H:%M:%S"))
print(tree(scan_dir))
time.sleep(8)
print()
print(time.strftime("%H:%M:%S"))
print(tree(scan_dir))

13:05:57
a8214d88-e269-482e-bbef-cd2f311f4ff8/
  control/  (0 files)
  events/  (2 files)
    .part-000001.parquet.tmp         0.11 MB
    part-000000.parquet              0.33 MB
  tdc/  (0 files)



13:06:05
a8214d88-e269-482e-bbef-cd2f311f4ff8/
  control/  (0 files)
  events/  (4 files)
    .part-000003.parquet.tmp         0.11 MB
    part-000000.parquet              0.33 MB
    part-000001.parquet              0.32 MB
    part-000002.parquet              0.32 MB
  tdc/  (1 files)
    .part-000000.parquet.tmp         0.00 MB


**A completed file is readable mid-scan.** Each `part-NNNNNN.parquet` is complete the moment it
has its final name; the file being written is the dot-prefixed `.part-NNNNNN.parquet.tmp`,
which parquet readers skip. Plain pandas reads the completed file:

In [4]:
completed = sorted((scan_dir / "events").glob("part-*.parquet"))
in_progress = sorted((scan_dir / "events").glob(".part-*.tmp"))
print("completed:  ", [p.name for p in completed])
print("in progress:", [p.name for p in in_progress])

df = pd.read_parquet(completed[0])
print(len(df), "rows;", df.dtypes.to_dict())
df.head()

completed:   ['part-000000.parquet', 'part-000001.parquet', 'part-000002.parquet']
in progress: ['.part-000003.parquet.tmp']
62369 rows; {'x': dtype('uint16'), 'y': dtype('uint16'), 'tot': dtype('uint16'), 't': dtype('uint64')}


,x,y,tot,t
0,115,156,103,38163066596737126
1,255,228,83,38163066638905400
2,32,151,73,38163066639202947
3,64,64,99,38163066639364079
4,51,8,98,38163066639482182


**Nothing is lost.** When the simulator disconnects the server publishes its stop message,
closes the files and writes `_scan_meta.json` last — its presence means the scan finished. The
row counts in the files equal the counts the simulator reports it sent, and `_scan_meta.json`
agrees with both.

In [5]:
scan_dir = demo.wait_scan_finished()
print(tree(scan_dir))

sent = demo.simulator_sent()
rows = {t: pq.ParquetDataset(scan_dir / t).read().num_rows for t in ("events", "tdc", "control")}
summary = json.loads((scan_dir / "_scan_meta.json").read_text())
sidecar = {t: summary["tables"][t]["num_rows"] for t in rows}

assert rows == sent, (rows, sent)
assert sidecar == rows, (sidecar, rows)
assert summary["status"] == "complete", summary["status"]
print("status:", summary["status"], "· dropped rows:", summary["dropped_rows"])
pd.DataFrame({"simulator sent": sent, "rows in parquet": rows, "_scan_meta.json": sidecar})

a8214d88-e269-482e-bbef-cd2f311f4ff8/
  _scan_meta.json                     2.5 kB
  control/  (1 files)
    part-000000.parquet              0.00 MB
  events/  (5 files)
    part-000000.parquet              0.33 MB
    part-000001.parquet              0.32 MB
    part-000002.parquet              0.32 MB
    part-000003.parquet              0.33 MB
    part-000004.parquet              0.29 MB
  tdc/  (1 files)
    part-000000.parquet              0.01 MB
status: complete · dropped rows: 0


,simulator sent,rows in parquet,_scan_meta.json
events,300443,300443,300443
tdc,3998,3998,3998
control,0,0,0


## 2. "Tiled should be able to read the file"

A Tiled server is started on a throw-away SQLite catalog with the same command as the
production unit in [`../tiled.md`](../tiled.md): `tiled serve catalog --api-key … --read
<output dir> catalog.db`. The scan is then registered with the function the UI calls when a
scan finishes (`tpx-tiled-register` on the command line). Registration **does not copy** the
data: Tiled is told where the parquet files are and serves them in place. The registrar reads
the first partition back through Tiled before it declares success.

In [6]:
uri = demo.start_tiled()
marker = demo.register(scan_dir)
print("Tiled at", uri)
print({k: marker[k] for k in ("action", "key", "status", "node_uri")})

2026-10-21 13:06:20,440 - INFO - Registered /tmp/tpx3_demo_1pmjo2i3/data/a8214d88-e269-482e-bbef-cd2f311f4ff8 as tpx3/a8214d88-e269-482e-bbef-cd2f311f4ff8 (created)


Tiled at http://127.0.0.1:59697
{'action': 'created', 'key': 'a8214d88-e269-482e-bbef-cd2f311f4ff8', 'status': 'complete', 'node_uri': 'http://127.0.0.1:59697/api/v1/metadata//tpx3/a8214d88-e269-482e-bbef-cd2f311f4ff8'}


**What pyarrow reads from disk and what Tiled serves over HTTP are the same table**, column for column:

In [7]:
client = demo.tiled_client()
node = client["tpx3"][marker["key"]]
print(node)

via_tiled = node["events"].read()                                    # whole table over HTTP
via_pyarrow = pq.ParquetDataset(scan_dir / "events").read().to_pandas()
print("partitions in Tiled:", node["events"].structure().npartitions, "= parquet files:", len(completed := sorted((scan_dir / "events").glob("part-*.parquet"))))
print("rows    tiled:", len(via_tiled), "  pyarrow:", len(via_pyarrow))
print("dtypes  tiled:", via_tiled.dtypes.to_dict())
print("dtypes pyarrow:", via_pyarrow.dtypes.to_dict())

order = ["t", "x", "y", "tot"]
pd.testing.assert_frame_equal(
    via_tiled.sort_values(order).reset_index(drop=True),
    via_pyarrow.sort_values(order).reset_index(drop=True),
)
print("identical.")
node["tdc"].read().head()

<Container {'events', 'tdc', 'control'}>


partitions in Tiled: 5 = parquet files: 5
rows    tiled: 300443   pyarrow: 300443
dtypes  tiled: {'x': dtype('uint16'), 'y': dtype('uint16'), 'tot': dtype('uint16'), 't': dtype('uint64')}
dtypes pyarrow: {'x': dtype('uint16'), 'y': dtype('uint16'), 'tot': dtype('uint16'), 't': dtype('uint64')}


identical.


,t,channel,edge
0,38163066651169805,1,0
1,38163066651170189,1,1
2,38163066672609624,1,0
3,38163066672610008,1,1
4,38163066710990361,1,0


## 3. "Taking metadata out of files, and having tiled put that metadata into its database"

Every parquet file carries the scan's metadata in its footer, as one JSON document under the
key `splash_timepix`: the acquisition settings, the operator fields, the data source, software
versions and, per file, the time range and row count. The registrar reads those footers (and
the row-group statistics) and hands the result to Tiled as the node's metadata, which Tiled
stores in its SQLite catalog and indexes for search. Nothing in the catalog came from anywhere
but the files.

In [8]:
footer = json.loads(pq.read_metadata(completed[0]).metadata[b"splash_timepix"])
print("footer of", completed[0].name, "— keys:", sorted(footer))
print(json.dumps({k: footer[k] for k in ("scan_name", "table", "file_index", "num_rows", "t_first", "t_last", "user", "source")}, indent=1))

footer of part-000000.parquet — keys: ['clock_hz', 'collapse_y', 'detector', 'detector_size_x', 'detector_size_y', 'dropped_rows_so_far', 'file_index', 'first_batch_wall_time_iso', 'hostname', 'mode', 'n_bins', 'num_rows', 'pyarrow_version', 'scan_name', 'schema_version', 'source', 'splash_timepix_version', 'start_time_iso', 't_cycle_ns', 't_delta_ns', 't_first', 't_first_s', 't_last', 't_last_s', 't_monotonic', 'table', 'tdc_channel', 'tdc_edge', 'tdc_frequency_hz', 'tot_clock_hz', 'user', 'writer_config']
{
 "scan_name": "a8214d88-e269-482e-bbef-cd2f311f4ff8",
 "table": "events",
 "file_index": 0,
 "num_rows": 62369,
 "t_first": 38163066596737126,
 "t_last": 38163082500997991,
 "user": {
  "sample": "Cu foil (demo)",
  "esaf": "demo-0001",
  "notes": "generated by demo_for_review.ipynb"
 },
 "source": {
  "kind": "simulator",
  "path": null
 }
}


**The footer and the Tiled node agree**; the node adds what only the whole scan knows (totals, overall time range, status):

In [9]:
md = dict(node.metadata)
compare = ["scan_name", "start_time_iso", "tdc_frequency_hz", "tdc_channel", "tdc_edge", "n_bins",
           "clock_hz", "tot_clock_hz", "user", "source", "splash_timepix_version"]
assert all(footer[k] == md[k] for k in compare), [k for k in compare if footer[k] != md[k]]
print("aggregated by the registrar:", {k: md[k] for k in ("num_events", "t_first", "t_last", "t_first_s", "t_last_s", "status")})
print("per table:", json.dumps(md["tables"]["events"], indent=1))
pd.DataFrame({"parquet footer": {k: footer[k] for k in compare}, "Tiled node metadata": {k: md[k] for k in compare}})

aggregated by the registrar: {'num_events': 300443, 't_first': 38163066596737126, 't_last': 38163143396553105, 't_first_s': 9938298.592900293, 't_last_s': 9938318.59285237, 'status': 'complete'}
per table: {
 "num_rows": 300443,
 "num_files": 5,
 "num_row_groups": 15,
 "bytes": 1592600,
 "files": [
  "part-000000.parquet",
  "part-000001.parquet",
  "part-000002.parquet",
  "part-000003.parquet",
  "part-000004.parquet"
 ],
 "t_first": 38163066596737126,
 "t_last": 38163143396553105,
 "t_first_s": 9938298.592900293,
 "t_last_s": 9938318.59285237
}


,parquet footer,Tiled node metadata
scan_name,a8214d88-e269-482e-bbef-cd2f311f4ff8,a8214d88-e269-482e-bbef-cd2f311f4ff8
start_time_iso,2026-10-21T20:05:51.140+00:00,2026-10-21T20:05:51.140+00:00
tdc_frequency_hz,100.0,100.0
tdc_channel,0,0
tdc_edge,rising,rising
n_bins,350,350
clock_hz,3840000000,3840000000
tot_clock_hz,40000000,40000000
user,"{'sample': 'Cu foil (demo)', 'esaf': 'demo-000...","{'sample': 'Cu foil (demo)', 'esaf': 'demo-000..."
source,"{'kind': 'simulator', 'path': None}","{'kind': 'simulator', 'path': None}"


**Searching the catalog.** Tiled answers these from its database, without opening a parquet
file. Nested keys (`user.sample`) and numeric ranges work because the footer is stored as JSON,
not as strings.

In [10]:
tpx3 = client["tpx3"]
t_mid = (md["t_first"] + md["t_last"]) // 2
print("sample == 'Cu foil (demo)'   :", list(tpx3.search(Key("user.sample") == "Cu foil (demo)")))
print("esaf == 'demo-0001'      :", list(tpx3.search(Key("user.esaf") == "demo-0001")))
print("sample == 'something else'   :", list(tpx3.search(Key("user.sample") == "something else")))
print("covering tick", t_mid, ":", list(tpx3.search(Key("t_first") <= t_mid).search(Key("t_last") >= t_mid)))
print("starting after the scan ended:", list(tpx3.search(Key("t_first") > md["t_last"])))
print("num_events >= 100000         :", list(tpx3.search(Key("num_events") >= 100_000)))
print("source.kind == 'simulator'   :", list(tpx3.search(Key("source.kind") == "simulator")))

sample == 'Cu foil (demo)'   : ['a8214d88-e269-482e-bbef-cd2f311f4ff8']


esaf == 'demo-0001'      : ['a8214d88-e269-482e-bbef-cd2f311f4ff8']
sample == 'something else'   : []
covering tick 38163104996645115 : ['a8214d88-e269-482e-bbef-cd2f311f4ff8']


starting after the scan ended: []


num_events >= 100000         : ['a8214d88-e269-482e-bbef-cd2f311f4ff8']
source.kind == 'simulator'   : ['a8214d88-e269-482e-bbef-cd2f311f4ff8']


**The metadata is in Tiled's database, not in the server's memory.** Stop the Tiled process,
start a new one on the same catalog file, and the searches still find the scan:

In [11]:
demo.restart_tiled()
client = demo.tiled_client()
tpx3 = client["tpx3"]
found = list(tpx3.search(Key("user.sample") == "Cu foil (demo)"))
print("after restart, sample == 'Cu foil (demo)':", found)
assert found == [marker["key"]]
print("after restart, partition 0 via Tiled:", len(tpx3[marker["key"]]["events"].read_partition(0)), "rows")

after restart, sample == 'Cu foil (demo)': ['a8214d88-e269-482e-bbef-cd2f311f4ff8']
after restart, partition 0 via Tiled: 62369 rows


## Clean-up

Stops `tpx-stream`, `tpx-sim` and the Tiled server and deletes the temporary directory.

In [12]:
demo.close()
print("done; removed", demo.workdir)

done; removed /tmp/tpx3_demo_1pmjo2i3


---

**What this notebook does not show** — and the acceptance checklist in
[`../README.md`](../README.md#7-acceptance-checklist) does: the real detector through
`live-cli` at the operational count rate, the detector snapshot that `acq.py` adds to the
metadata, the UI's own registration and start-up catch-up, and the measured bytes per hit and
disk bandwidth that justify parquet being on by default.